# Classification Templates (Datasets 1–4)

Edit the paths under **Config** and run cells top-to-bottom for each dataset. 
Outputs are saved into the `results/` folder with your last name in the filename.


In [48]:
# ==== Config (edit me) ====
LAST_NAME = "Arowolo"  # <-- replace with your last name
DATA_DIR = "../data" # folder containing TrainData*.txt, TrainLabel*.txt, TestData*.txt

# Expected filenames (edit if your files differ)
FILES = {
    1: {"X_train": "TrainData1.txt", "y_train": "TrainLabel1.txt", "X_test": "TestData1.txt"},
    2: {"X_train": "TrainData2.txt", "y_train": "TrainLabel2.txt", "X_test": "TestData2.txt"},
    3: {"X_train": "TrainData3.txt", "y_train": "TrainLabel3.txt", "X_test": "TestData3.txt"},
    4: {"X_train": "TrainData4.txt", "y_train": "TrainLabel4.txt", "X_test": "TestData4.txt"},
}

RESULTS_DIR = "../results"
MISSING_SENTINEL = 1.00000000000000e+99  # given in the assignment
RANDOM_STATE = 42


In [49]:
# ==== Imports ====
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score


In [50]:
# ==== Utilities ====
def load_matrix_txt(path: str) -> pd.DataFrame:
    """Loads a numeric matrix from txt/csv without headers. Handles whitespace or comma separated."""
    # Try whitespace, then comma
    try:
        df = pd.read_csv(path, sep=None, header=None, engine='python')
    except Exception:
        df = pd.read_csv(path, sep='\s+', header=None, engine='python')
    return df

def load_labels_txt(path: str) -> pd.Series:
    # Read any mix of whitespace/commas into a 2-D frame
    df = pd.read_csv(path, sep=r"[\s,]+", header=None, engine="python")
    # Flatten row-wise -> 1-D
    vals = pd.to_numeric(df.values.ravel(order="C"), errors="coerce")
    # Drop blanks/NAs and cast to int
    vals = pd.Series(vals).dropna().astype(int).reset_index(drop=True)
    return vals


def replace_missing(df: pd.DataFrame, sentinel=MISSING_SENTINEL) -> pd.DataFrame:
    # Replace sentinel with NaN (float) for imputation later
    out = df.replace(sentinel, np.nan)
    return out

def model_candidates(random_state=RANDOM_STATE):
    return {
        "LogReg": LogisticRegression(max_iter=2000, multi_class="auto", random_state=random_state, n_jobs=None),
        "SVM_RBF": SVC(kernel="rbf", probability=True, random_state=random_state),
        "RF": RandomForestClassifier(n_estimators=300, random_state=random_state, n_jobs=-1),
        "KNN": KNeighborsClassifier(n_neighbors=5)
    }

def make_numeric_pipeline_for(tabular_df: pd.DataFrame):
    """Impute + scale for numeric features. Scaling helps SVM/KNN."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False) if pd.api.types.is_sparse(tabular_df) else StandardScaler())
    ])

def evaluate_models(X: pd.DataFrame, y: pd.Series, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    results = {}
    for name, clf in model_candidates().items():
        pipe = Pipeline([
            ("impute_scale", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("clf", clf)
        ])
        scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
        results[name] = (scores.mean(), scores.std())
    return results

def best_model_by_cv(X: pd.DataFrame, y: pd.Series):
    res = evaluate_models(X, y, cv=5)
    best_name = max(res, key=lambda k: res[k][0])
    # Construct the best pipeline to fit on full data
    clf = model_candidates()[best_name]
    best_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", clf)
    ])
    return best_name, best_pipe, res


<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
/var/folders/3g/76lf4vrx7vd2xb8bm15rdkz80000gn/T/ipykernel_95921/3364802582.py:8: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(path, sep='\s+', header=None, engine='python')


## Dataset Runner
Run this cell per dataset index (1 through 4). It will:
1) Load train/test
2) Replace missing values
3) Cross-validate candidate models
4) Fit best on full data
5) Save predictions to `results/{LAST_NAME}Classification{i}.txt`

In [51]:
def run_dataset(i: int):
    assert i in FILES, f"Dataset index {i} not in FILES mapping."
    f = FILES[i]
    Xtr_path = os.path.join(DATA_DIR, f["X_train"])
    ytr_path = os.path.join(DATA_DIR, f["y_train"])
    Xte_path = os.path.join(DATA_DIR, f["X_test"])
    
    Xtr = load_matrix_txt(Xtr_path)
    ytr = load_labels_txt(ytr_path)
    Xte = load_matrix_txt(Xte_path)

    # After loading Xtr, ytr
    if len(ytr) != len(Xtr):
        n = min(len(Xtr), len(ytr))
        print(f"Warning: Xtr has {len(Xtr)} rows but ytr has {len(ytr)} labels. Using first {n}.")
        Xtr = Xtr.iloc[:n, :].reset_index(drop=True)
        ytr = ytr.iloc[:n].reset_index(drop=True)

    
    # Replace sentinel missing values
    Xtr = replace_missing(Xtr, MISSING_SENTINEL)
    Xte = replace_missing(Xte, MISSING_SENTINEL)
    
    # Cross-validate candidates and pick best
    best_name, best_pipe, cv_results = best_model_by_cv(Xtr, ytr)
    print(f"[Dataset {i}] CV mean accuracies:")
    for k,(m,s) in cv_results.items():
        print(f"  {k}: {m:.4f} ± {s:.4f}")
    print(f"Selected: {best_name}")
    
    # Fit on all training data and predict
    best_pipe.fit(Xtr, ytr)
    y_pred = best_pipe.predict(Xte).astype(int)
    
    # Save
    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, f"{LAST_NAME}Classification{i}.txt")
    pd.Series(y_pred).to_csv(out_path, header=False, index=False)
    print(f"Saved predictions to {out_path}")
    
# Example usage (uncomment and run):
run_dataset(1)
run_dataset(2)
run_dataset(3)
run_dataset(4)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247:

[Dataset 1] CV mean accuracies:
  LogReg: 0.9667 ± 0.0365
  SVM_RBF: 0.9067 ± 0.0389
  RF: 0.9067 ± 0.0389
  KNN: 0.9333 ± 0.0516
Selected: LogReg


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Saved predictions to ../results/ArowoloClassification1.txt


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:598: UserWarning: Skipping features without any observed values: [    0     1     2 ... 27542 27544 27545]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:598: UserWarning: Skipping features without any observed values: [    0     1     2 ... 27542 27544 27545]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:598: UserWarning: Skipping features without any observed values: [    0     1     2 ... 27542 27544 27545]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-pa

...

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Pytho

[Dataset 3] CV mean accuracies:
  LogReg: 0.8547 ± 0.0137
  SVM_RBF: 0.9144 ± 0.0077
  RF: 0.9639 ± 0.0045
  KNN: 0.8378 ± 0.0154
Selected: RF
Saved predictions to ../results/ArowoloClassification3.txt


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:598: UserWarning: Skipping features without any observed values: [ 0  1  2  4  5  7  8 10 11 13 14 16 17 19 20 22 23 25 26 28 29 31 32]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:598: UserWarning: Skipping features without any observed values: [ 0  1  2  4  5  7  8 10 11 13 14 16 17 19 20 22 23 25 26 28 29 31 32]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(

[Dataset 4] CV mean accuracies:
  LogReg: 0.5996 ± 0.0167
  SVM_RBF: 0.6014 ± 0.0226
  RF: 0.6756 ± 0.0310
  KNN: 0.5541 ± 0.0186
Selected: RF
Saved predictions to ../results/ArowoloClassification4.txt


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:598: UserWarning: Skipping features without any observed values: [ 0  1  2  4  5  7  8 10 11 13 14 16 17 19 20 22 23 25 26 28 29 31 32]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
